In [ ]:
import pandas as pd
import requests
import json

# DHIS2 API details
dhis2_url = 'https://impilo-dhis.mohcc.gov.zw/api/dataValueSet'
dhis2_username = ''
dhis2_password = ''

# CSV file
csv_file = '.csv'

# Read CSV file
try:
    data = pd.read_csv(csv_file)
except FileNotFoundError:
    print(f"File not found: {csv_file}")
    exit(1)

# Check for required columns
required_columns = ['DataElement', 'Period', 'OrgUnit', 'Value']
missing_columns = [col for col in required_columns if col not in data.columns]
if missing_columns:
    print(f"Missing required columns in CSV: {missing_columns}")
    exit(1)

# Transform data into DHIS2 JSON format
data_values = []
for index, row in data.iterrows():
    data_value = {
        "dataElement": row['DataElement'],
        "period": row['Period'],
        "orgUnit": row['OrgUnit'],
        "value": row['Value'],
        "categoryOptionCombo": row['CategoryOptionCombo'],
        # Handle NaN values in AttributeOptionCombo
        "attributeOptionCombo": row['AttributeOptionCombo'] if pd.notna(row['AttributeOptionCombo']) else None
    }
    data_values.append(data_value)

json_payload = {
    "dataValues": data_values
}

# Send data to DHIS2
try:
    response = requests.post(dhis2_url, auth=(dhis2_username, dhis2_password), json=json_payload)
    if response.status_code == 200:
        print("Data uploaded successfully.")
    else:
        # Print detailed error message from DHIS2
        print(f"Response from DHIS2: {response.text}")
except requests.exceptions.HTTPError as errh:
    print(f"HTTP Error: {errh}")
    print(f"Response from DHIS2: {response.text}")
except requests.exceptions.ConnectionError as errc:
    print(f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print(f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print(f"Error: {err}")